# What Is Artificial Intelligence (AI)?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-builders/curriculum/blob/main/notebooks/01_ml_what_2026.ipynb)

## What You'll Learn

Welcome to **What is ~~Machine Learning~~ Artificial Intelligence (AI)?**, Lesson 1 of the AI Builders curriculumー2026 edition.

In this notebook we will cover:

1. **AI / ML / DL / LLM definitions**; what each term means and how they nest inside each other

2. **Three paradigms for building AI systems**; Rule-based, Machine Learning, and LLM Systems

3. **Hands-on LLM System demo**; you will first define a task as a DSPy signature and iterate on the instruction by hand, then watch DSPy's optimizer rewrite it automatically

4. **Training loop**; the core mechanism behind how ML models learn from data

5. **Traditional ML demos**; neural network regression (linear vs non-linear curve fitting) and image classification (Thai food with FoodyDudy)

6. **Evaluation**; metrics, train/validation/test splits, and a preview of modern LLM evaluation challenges

We start with what is most familiar to you, LLMs and prompts, then peel back the layers to see what is happening under the hood.

## Environment Setup

The cell below detects whether you are running on **Google Colab** or **locally**.

- **Colab**: dependencies are installed automatically via `pip`.
- **Local** (e.g. MacBook Pro with Apple Silicon): set up your environment with `uv`:

```bash
uv venv && uv pip install dspy torch transformers bitsandbytes accelerate fastai pytorch-lightning matplotlib datasets
```

Or if you have a `requirements.txt`:

```bash
uv venv && uv pip install -r requirements.txt
```

In [8]:
import os, subprocess, sys

# --- Detect environment ---
IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    print("Running on Google Colab — installing dependencies...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q",
         "dspy", "torch", "transformers", "bitsandbytes",
         "accelerate", "fastai", "pytorch-lightning",
         "matplotlib", "datasets"]
    )
    print("Done!")
else:
    print("Running locally. Make sure you've set up your environment:")
    print("  uv venv && uv pip install dspy torch transformers bitsandbytes accelerate fastai pytorch-lightning matplotlib datasets")
    print("Or: uv pip install -r requirements.txt")

# --- Device setup ---
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

print(f"\nDevice: {DEVICE}")

Running locally. Make sure you've set up your environment:
  uv venv && uv pip install dspy torch transformers bitsandbytes accelerate fastai pytorch-lightning matplotlib datasets
Or: uv pip install -r requirements.txt
Using Apple Silicon MPS

Device: mps


## AI / ML / DL / LLM — What Do These Terms Actually Mean?

You have probably seen these acronyms everywhere. Let's pin down what each one means and how they relate to each other.

**Artificial Intelligence (AI)** is the broadest term — it covers any system designed to think or act intelligently. This includes everything from a chess engine to a self-driving car to a chatbot.

**Machine Learning (ML)** is a subset of AI. Instead of a human writing explicit rules, an ML system *learns* the rules from data. You give it examples, and it figures out the patterns.

**Deep Learning (DL)** is a subset of ML that uses **multi-layer neural networks** — models with many stacked layers of simple computations that can learn very complex patterns. Most of the breakthroughs you hear about in image recognition, speech, and language come from deep learning.

**Large Language Models (LLMs)** are a subset of DL — large-scale neural networks trained on massive amounts of text to do one deceptively simple thing: **predict the next token**. GPT, Qwen, Llama, and Claude are all LLMs.

Here is the key nuance: while LLMs *are* deep learning models, the way we *use* them is fundamentally different. We don't retrain them for each task — we **prompt** them. This prompting paradigm is so different that it deserves its own category, which we cover in the next section.

The nested relationship looks like this:

```
┌─────────────────────────────────────────────────────┐
│  Artificial Intelligence (AI)                       │
│  Systems that think or act intelligently            │
│                                                     │
│  ┌───────────────────────────────────────────────┐  │
│  │  Machine Learning (ML)                        │  │
│  │  Rules learned from data                      │  │
│  │                                               │  │
│  │  ┌─────────────────────────────────────────┐  │  │
│  │  │  Deep Learning (DL)                     │  │  │
│  │  │  Multi-layer neural networks            │  │  │
│  │  │                                         │  │  │
│  │  │  ┌───────────────────────────────────┐  │  │  │
│  │  │  │  Large Language Models (LLMs)     │  │  │  │
│  │  │  │  Trained on text to predict       │  │  │  │
│  │  │  │  the next token                   │  │  │  │
│  │  │  └───────────────────────────────────┘  │  │  │
│  │  └─────────────────────────────────────────┘  │  │
│  └───────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────┘
```

Each box lives entirely inside the one above it: every LLM is a DL model, every DL model is an ML model, and every ML model is a form of AI.

## Three Paradigms of AI Systems

Now that we know what these terms mean, let's look at the three major ways people actually **build** AI systems.

### 1. Rule-based Systems

A human expert writes explicit rules that transform inputs into outputs. Think of a spam filter with hand-crafted rules like *"if the email contains 'FREE MONEY', mark it as spam."* The human defines every rule — the system just follows them.

```
  Human Expert
       │
       │ writes
       v
  ┌─────────┐       ┌─────────┐
  │  Rules  │──────>│ Output  │
  └─────────┘       └─────────┘
       ^                        
       │                        
  ┌─────────┐                   
  │  Input  │                   
  └─────────┘                   
```

### 2. Machine Learning Systems

Instead of writing rules by hand, you collect **historical input-output pairs** (training data) and let an algorithm learn the rules (a model) from that data. A spam classifier trained on thousands of labeled emails learns its own rules — rules that are often better than what a human could write.

```
  ┌─────────┐  ┌─────────┐
  │  Input  │  │ Output  │    Training Data
  └────┬────┘  └────┬────┘    (historical pairs)
       │            │
       v            v
  ┌──────────────────────┐
  │   Learning Algorithm │    learns rules from data
  └──────────┬───────────┘
             │
             v
  ┌──────────────────────┐
  │    Model (Rules)     │    Input ──> Model ──> Output
  └──────────────────────┘
```

### 3. LLM Systems

You take a **pre-trained LLM** and give it a task via a **prompt** — zero-shot, few-shot, or with optimized instructions. You don't train a task-specific model from scratch. You don't even need labeled training data for many tasks. You write a prompt, test it, iterate, and ship.

```
  ┌─────────┐   ┌──────────────┐
  │  Input  │   │    Prompt    │    written by human or optimizer
  └────┬────┘   └──────┬───────┘
       │               │
       v               v
  ┌──────────────────────────┐
  │   Pre-trained LLM        │    already knows language
  └────────────┬─────────────┘
               │
               v
  ┌──────────────────────────┐
  │         Output           │
  └──────────────────────────┘
```

### The Paradigm Shift

LLM Systems represent a genuine paradigm shift. Building with LLMs is closer to **software engineering** than traditional ML — you compose and prompt rather than collect data and train. The skills you need look more like writing clear instructions and designing good evaluations than feature engineering and gradient tuning.

Here is a side-by-side comparison:

| | **Rule-based** | **ML Systems** | **LLM Systems** |
|---|---|---|---|
| **Who defines the rules?** | Human expert | Learned from data | Pre-trained model + prompt |
| **What data is needed?** | Domain knowledge | Labeled input-output pairs | (Optional) few examples in the prompt |
| **How is the system built?** | Write rules manually | Collect data → train model | Write prompt → test → iterate |
| **Closest analogy** | Traditional programming | Statistical modeling | Software engineering |
| **Example** | `if temperature > 30: "hot"` | Train a classifier on weather data | Ask an LLM: *"Classify this weather reading"* |

Of course, these paradigms are NOT mutually exclusive. Some ML systems apply rules on top of their predictions; some LLM systems finetune their weights using additional training examples.

In the next sections, we will get hands-on with LLM Systems first, and then peel back the layers to understand the ML training loop underneath.

## LLM System Demo — From Programming to Optimization

Time to get hands-on. We are going to build an LLM System for **reading comprehension** using the [SQuAD](https://rajpurkar.github.io/SQuAD-explorer/) dataset.

The task: given a context paragraph and a question, extract the correct answer from the context.

Instead of writing raw prompts, we will use [DSPy](https://dspy.ai/) — a framework that treats LLM systems as **programs**, not prompt strings. You define *what* the task is (a signature), and DSPy handles *how* to talk to the model.

Here is the plan:

1. Load data (train / validation / test splits)
2. Load an LLM (Qwen3-0.6B)
3. Define the task as a DSPy signature — **you edit the instruction, re-run, see score change**
4. Let DSPy's optimizer rewrite the instruction automatically
5. Compare your manual score vs. the optimizer's score

### Step 1: Load the SQuAD Dataset

We load [SQuAD v1](https://huggingface.co/datasets/rajpurkar/squad) from HuggingFace. Each example has a context paragraph, a question, and an answer.

We sample from three different topics so there is zero overlap between splits:
- **Training** (20 examples from *European Union Law*) — the optimizer's study material
- **Validation** (5 examples from *Imperialism*) — used during optimization to pick the best candidate
- **Test** (5 examples from *Chloroplast*) — the final exam, never seen during optimization

In [9]:
import textwrap, random
from datasets import load_dataset

ds = load_dataset("rajpurkar/squad", split="validation")

def sample_by_title(dataset, title, n, seed=42):
    pool = [ex for ex in dataset if ex["title"] == title]
    chosen = random.Random(seed).sample(pool, min(n, len(pool)))
    return [{"question": ex["question"], "context": ex["context"], "answer": ex["answers"]["text"][0]} for ex in chosen]

train_examples = sample_by_title(ds, "European_Union_law", 20)
val_examples   = sample_by_title(ds, "Imperialism", 5)
test_examples  = sample_by_title(ds, "Chloroplast", 5)

print(f"Train: {len(train_examples)} (European Union Law)")
print(f"Val:   {len(val_examples)} (Imperialism)")
print(f"Test:  {len(test_examples)} (Chloroplast)")
print(f"\nSample: {test_examples[0]['question']}")
print(f"Answer: {test_examples[0]['answer']}")

Train: 20 (European Union Law)
Val:   5 (Imperialism)
Test:  5 (Chloroplast)

Sample: What part of cryptophyte chloroplasts is similar to chlorarachniophytes?
Answer: nucleomorph


### Step 2: Load Qwen3-0.6B and Configure DSPy

We load **Qwen3-0.6B** with 4-bit quantization, then wrap it for DSPy.

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, re

model_name = "Qwen/Qwen3-0.6B"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=quantization_config, device_map="auto",
)
print("Model loaded!")

Loading Qwen/Qwen3-0.6B...


Loading weights:   1%|          | 2/311 [00:00<02:14,  2.30it/s]/Users/charipol/Work/curriculum/.venv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 311/311 [00:05<00:00, 58.00it/s]


Model loaded!


In [11]:
def strip_thinking(text):
    """Remove Qwen3's <think>...</think> reasoning block from output."""
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<think>.*', '', text, flags=re.DOTALL)
    return text.strip()

In [12]:
import dspy

class LocalQwen(dspy.BaseLM):
    """Minimal wrapper: reuse our already-loaded HF model for DSPy."""
    def __init__(self, hf_model, hf_tokenizer, **kwargs):
        super().__init__(model="local-qwen3-0.6b", **kwargs)
        self.hf_model = hf_model
        self.hf_tokenizer = hf_tokenizer

    def forward(self, prompt=None, messages=None, **kwargs):
        if messages is None:
            messages = [{"role": "user", "content": prompt}]
        text = self.hf_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        inputs = self.hf_tokenizer(text, return_tensors="pt").to(self.hf_model.device)
        input_len = inputs["input_ids"].shape[1]
        max_tok = kwargs.get("max_tokens") or self.kwargs.get("max_tokens", 256)
        with torch.no_grad():
            out = self.hf_model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
        answer = self.hf_tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        answer = strip_thinking(answer)

        #Just boilerplate to get the right format; don't worry about it. Leave it to AI
        import time
        from openai.types.chat import ChatCompletion, ChatCompletionMessage
        from openai.types.chat.chat_completion import Choice
        from openai.types import CompletionUsage
        n_gen = len(out[0]) - input_len
        return ChatCompletion(
            id=f"local-{int(time.time())}", created=int(time.time()),
            model="local-qwen3-0.6b", object="chat.completion",
            choices=[Choice(finish_reason="stop", index=0,
                            message=ChatCompletionMessage(role="assistant", content=answer))],
            usage=CompletionUsage(prompt_tokens=input_len, completion_tokens=n_gen,
                                  total_tokens=input_len + n_gen),
        )

lm = LocalQwen(model, tokenizer, max_tokens=128)
dspy.configure(lm=lm)
print("DSPy configured!")

DSPy configured!


### Step 3: Define the Task as a DSPy Signature and Module

In DSPy, you build LLM systems with two building blocks:

- A [**Signature**](https://dspy.ai/learn/programming/signatures/) declares *what* the task is — a Python class with a docstring (the instruction) and typed input/output fields.
- A [**Module**](https://dspy.ai/learn/programming/modules/) declares *how* to solve it — it wraps one or more signatures into a program. The simplest module uses `dspy.Predict` (just call the LLM once), but you could swap in `dspy.ChainOfThought` to make the model reason step-by-step.

The docstring is the *"weight"* you will tune. Edit it, re-run, see if the score changes.

> Some ideas to try:
> 1. **Be specific about instruction and field descriptions** — "Answer the question" or "the context" are generic. "Extract the shortest span from the context that answers the question" or "a paragraph that contains the answer" tells the model exactly what you want. The more precise your instruction, the less room the model has to go off track.
> 2. **Constrain the output format** — Small models tend to ramble. Change the answer `desc` to something like "1-5 words copied directly from the context". When you tell the model *how* to answer (not just *what* to answer), you get more predictable results.
> 3. **Use the right verb** — "Answer" invites paraphrasing. "Extract" or "Copy" tells the model to pull words directly from the context. Word choice matters — the model takes your instruction literally.
> 4. **Give the model a role** — Try adding "You are a reading comprehension system" to the docstring. Framing the task with a role can nudge the model's behavior.
> 5. **Think about what went wrong** — Look at the MISS cases. Did the model paraphrase when it should have quoted? Did it give too long an answer? Did it hallucinate? Each failure mode suggests a different fix to the instruction.


In [13]:
# ============================================================
# YOUR SIGNATURE — edit the docstring and field descs, then
# re-run this cell and the evaluation cell to see your score!
# ============================================================

#what to do
class SquadQA(dspy.Signature):
    """Answer the question"""
    context = dspy.InputField(desc="the context")
    question = dspy.InputField(desc="the question about the context")
    answer = dspy.OutputField(desc="the answer to the question")

#how to do it
class SquadQAModule(dspy.Module):
    def __init__(self):
        self.predict = dspy.Predict(SquadQA)

    def forward(self, context, question):
        return self.predict(context=context, question=question)


qa_module = SquadQAModule()
print("Signature + Module defined! Run the next cell to evaluate.")

Signature + Module defined! Run the next cell to evaluate.


In [14]:
def answer_hit(prediction, gold):
    """1.0 if gold answer appears in prediction (case-insensitive), else 0.0."""
    return 1.0 if gold.lower() in prediction.lower() else 0.0

def dspy_answer_hit(example, prediction, trace=None):
    """DSPy-compatible metric."""
    return answer_hit(prediction.answer, example.answer)

# Wrap data for DSPy
trainset = [dspy.Example(**ex).with_inputs("context", "question") for ex in train_examples]
valset   = [dspy.Example(**ex).with_inputs("context", "question") for ex in val_examples]
testset  = [dspy.Example(**ex).with_inputs("context", "question") for ex in test_examples]

# Evaluate on test set
evaluator = dspy.Evaluate(
    devset=testset,
    metric=dspy_answer_hit,
    num_threads=1,
    display_progress=True,
)

eval_result = evaluator(qa_module)
my_score = eval_result.score
print(f"\n=== Your score: {my_score:.1f}% ===")

Average Metric: 3.00 / 5 (60.0%): 100%|██████████| 5/5 [01:31<00:00, 18.38s/it]

2026/03/31 22:10:49 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 5 (60.0%)




=== Your score: 60.0% ===


In [15]:
#what the model actually answered
for i, (example, prediction, score) in enumerate(eval_result.results):
    status = "HIT" if score > 0 else "MISS"
    print(f"[{i+1}/{len(eval_result.results)}] {status}")
    print(f"  Q:    {example.question}")
    print(f"  Gold: {example.answer}")
    print(f"  Pred: {prediction.answer}")
    print()

[1/5] MISS
  Q:    What part of cryptophyte chloroplasts is similar to chlorarachniophytes?
  Gold: nucleomorph
  Pred: The part of the cryptophyte chloroplasts that is similar to chlorarachniophytes is the pyrenoid and thylakoids in stacks of two.

[2/5] HIT
  Q:    What was Konstantin Mereschkowski's career?
  Gold: biologist
  Pred: Konstantin Mereschkowski was a Russian biologist.

[3/5] MISS
  Q:    What does the inner mitochondria membrane do?
  Gold: run proton pumps and carry out oxidative phosphorylation
  Pred: The inner mitochondria membrane is responsible for the transport of protons across the mitochondrial membrane, enabling oxidative phosphorylation to generate ATP.

[4/5] HIT
  Q:    What new tasks do the protein products of transferred genes take on?
  Gold: participating in cell division, protein routing, and even disease resistance
  Pred: The protein products of transferred genes take on new tasks such as participating in cell division, protein routing, and even dis

### What Prompt Did DSPy Actually Send?

Let's peek at the exact prompt DSPy constructed from your signature. This is what the LLM actually saw.

In [16]:
# The last LLM call — see the full prompt
last = lm.history[-1]
for msg in last['messages']:
    print(textwrap.fill(f"[{msg['role']}]",width=80))
    print(textwrap.fill(msg['content'],width=80))
    print()

[system]
Your input fields are: 1. `context` (str): the context 2. `question` (str): the
question about the context Your output fields are: 1. `answer` (str): the answer
to the question All interactions will be structured in the following way, with
the appropriate values filled in.  [[ ## context ## ]] {context}  [[ ## question
## ]] {question}  [[ ## answer ## ]] {answer}  [[ ## completed ## ]] In adhering
to this structure, your objective is:          Answer the question

[user]
[[ ## context ## ]] In cpDNA, there are several A → G deamination gradients. DNA
becomes susceptible to deamination events when it is single stranded. When
replication forks form, the strand not being copied is single stranded, and thus
at risk for A → G deamination. Therefore, gradients in deamination indicate that
replication forks were most likely present and the direction that they initially
opened (the highest gradient is most likely nearest the start site because it
was single stranded for the longest a

### You Are the Optimizer

Take a moment to reflect on what just happened:

- The **signature docstring** is your weight — it controls how the model behaves on the task.
- The **test set** tells you how well your current weight is doing.
- The **hit rate** is your metric — it gives you a number to optimize.
- **You** are the optimizer — you read the score, think about what went wrong, edit the docstring, and re-run.

Go back to the signature cell, change the docstring or field descriptions, re-run the evaluation, and see if your score improves.

In the next section, we will let DSPy's optimizer do this search automatically.

## LLM System Demo — Automatic Prompt Optimization

You just experienced prompt optimization by hand — editing the signature docstring, testing, tweaking, repeat.

Now let's let [**BootstrapFewShot**](https://dspy.ai/learn/optimization/optimizers/#automatic-few-shot-learning) do something you can't easily do by hand: automatically find good **few-shot examples** from the training data and add them to the prompt.

BootstrapFewShot works like this:

1. Run the model on training examples
2. Keep the examples where the model got the right answer
3. Add those successful examples to the prompt as demonstrations

The result: the model sees worked examples before answering each new question — like a student studying solved problems before an exam.

> Some ideas to try: Play with other [optimizers](https://dspy.ai/learn/optimization/optimizers/) like MIPROv2 and GEPA.


In [ ]:
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=dspy_answer_hit,
    max_bootstrapped_demos=3,  # up to 3 examples found by running the model
    max_labeled_demos=3,       # up to 3 examples taken directly from training data
    max_rounds=1,
)

print("Running BootstrapFewShot optimizer...")
optimized_module = optimizer.compile(
    student=SquadQAModule(), #fresh copy of module to be optimized
    trainset=trainset, #augment the prompt with examples from training set
)
print("Optimization complete!")

Running BootstrapFewShot optimizer...


 50%|█████     | 10/20 [01:53<01:53, 11.31s/it]

Bootstrapped 3 full traces after 10 examples for up to 1 rounds, amounting to 10 attempts.
Optimization complete!


### What Did the Optimizer Change?

Let's look at the actual prompt before and after optimization. The difference is the few-shot examples that BootstrapFewShot added.

In [21]:
# Capture ORIGINAL prompt
_ = qa_module(context=testset[0].context, question=testset[0].question)
original_messages = lm.history[-1]['messages']

# Capture OPTIMIZED prompt  
_ = optimized_module(context=testset[0].context, question=testset[0].question)
optimized_messages = lm.history[-1]['messages']

print("=" * 80)
print(f"BEFORE OPTIMIZATION ({len(original_messages)} messages)")
print("=" * 80)
for msg in original_messages:
    print(f"\n[{msg['role']}]")
    print(textwrap.fill(msg['content'], width=80))
    
print("\n" + "=" * 80)
print(f"AFTER OPTIMIZATION ({len(optimized_messages)} messages)")
print("=" * 80)
for msg in optimized_messages:
    print(f"\n[{msg['role']}]")
    print(textwrap.fill(msg['content'], width=80))

print(f"\nBefore: {len(original_messages)} messages (just the question)")
print(f"After:  {len(optimized_messages)} messages ({(len(optimized_messages)-2)//2} few-shot demos + the question)")

BEFORE OPTIMIZATION (2 messages)

[system]
Your input fields are: 1. `context` (str): the context 2. `question` (str): the
question about the context Your output fields are: 1. `answer` (str): the answer
to the question All interactions will be structured in the following way, with
the appropriate values filled in.  [[ ## context ## ]] {context}  [[ ## question
## ]] {question}  [[ ## answer ## ]] {answer}  [[ ## completed ## ]] In adhering
to this structure, your objective is:          Answer the question

[user]
[[ ## context ## ]] Cryptophytes, or cryptomonads are a group of algae that
contain a red-algal derived chloroplast. Cryptophyte chloroplasts contain a
nucleomorph that superficially resembles that of the chlorarachniophytes.
Cryptophyte chloroplasts have four membranes, the outermost of which is
continuous with the rough endoplasmic reticulum. They synthesize ordinary
starch, which is stored in granules found in the periplastid space—outside the
original double membrane, in 

### Evaluate the Optimized Module

Same test set, same metric — but now the prompt includes few-shot examples.

In [23]:
optimized_eval_result = evaluator(optimized_module)
optimized_score = optimized_eval_result.score
print(f"\n=== Optimized score: {optimized_score:.1f}% ===")

#what the model actually answered
for i, (example, prediction, score) in enumerate(optimized_eval_result.results):
    status = "HIT" if score > 0 else "MISS"
    print(f"[{i+1}/{len(optimized_eval_result.results)}] {status}")
    print(f"  Q:    {example.question}")
    print(f"  Gold: {example.answer}")
    print(f"  Pred: {prediction.answer}")
    print()

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/charipol/Work/curriculum/.venv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py:314: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Average Metric: 5.00 / 5 (100.0%): 100%|██████████| 5/5 [01:31<00:00, 18.38s/it]

2026/03/31 22:25:31 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 5 (100.0%)




=== Optimized score: 100.0% ===
[1/5] HIT
  Q:    What part of cryptophyte chloroplasts is similar to chlorarachniophytes?
  Gold: nucleomorph
  Pred: The part of the cryptophyte chloroplasts similar to chlorarachniophytes is the nucleomorph.

[2/5] HIT
  Q:    What was Konstantin Mereschkowski's career?
  Gold: biologist
  Pred: Konstantin Mereschkowski was a Russian biologist who made significant contributions to the understanding of the origins of chloroplasts. He is known for his work on endosymbiosis, which explains how eukaryotic cells like chloroplasts evolved from photosynthetic bacteria.

[3/5] HIT
  Q:    What does the inner mitochondria membrane do?
  Gold: run proton pumps and carry out oxidative phosphorylation
  Pred: The inner mitochondria membrane is used to run proton pumps and carry out oxidative phosphorylation across to generate ATP energy.

[4/5] HIT
  Q:    What new tasks do the protein products of transferred genes take on?
  Gold: participating in cell divisio

### Compare — Manual vs. Optimized

In [24]:
print("=" * 50)
print("  SCORE COMPARISON — SQuAD Answer Hit Rate")
print("=" * 50)
print(f"  Your signature:          {my_score:.1f}%")
print(f"  + BootstrapFewShot:      {optimized_score:.1f}%")
print("=" * 50)

if optimized_score > my_score:
    print(f"\n  Few-shot demos improved the score by {optimized_score - my_score:.1f}%!")
    print("  The optimizer found training examples that teach the model by demonstration.")
elif optimized_score < my_score:
    print(f"\n  Your signature was already better by {my_score - optimized_score:.1f}%!")
else:
    print("\n  Tied!")

  SCORE COMPARISON — SQuAD Answer Hit Rate
  Your signature:          60.0%
  + BootstrapFewShot:      100.0%

  Few-shot demos improved the score by 40.0%!
  The optimizer found training examples that teach the model by demonstration.


### DSPy Concepts Map to the Training Loop

What DSPy just did looks a lot like *training*. The concepts map directly:

| **DSPy Concept** | **Training Loop Concept** | **What it does** |
|---|---|---|
| Signature docstring + demos | **Weights** | The learnable parameters that control model behavior |
| BootstrapFewShot | **Optimizer** | The algorithm that updates the parameters to improve performance |
| `dspy_answer_hit` metric | **Loss / Metric** | The signal that tells the optimizer how well the current parameters are doing |
| `trainset` (SQuAD examples) | **Training Data** | The examples the optimizer uses to search for better parameters |

The key insight: **prompt optimization is a form of training**. The "weights" are text (instructions and few-shot examples) instead of floating-point numbers, and the "optimizer" searches over discrete prompt variations instead of following gradients — but the structure is the same.

```
Traditional ML:   Data → Model(weights) → Predictions → Loss → Optimizer → Updated weights
DSPy:             Data → Module(prompt)  → Predictions → Metric → Optimizer → Updated prompt
```

We will formalize the training loop in detail in the next section. But now you have the intuition: you have already *done* training — first by hand, then with BootstrapFewShot.

## What's Under the Hood?

Let's take a step back.

You just built an LLM System two ways: first by hand (writing and tweaking prompts), then with DSPy (letting an optimizer search for better prompts automatically). In both cases, the thing being optimized was **text** — the prompt instructions and few-shot examples.

But here is the thing: the LLM you were prompting (Qwen3-0.6B) is itself a neural network with **hundreds of millions of numerical parameters** — weights — that were trained on massive amounts of text. Someone had to train those weights before you could prompt the model. How did that happen?

The answer is the **training loop** — the core mechanism behind all of machine learning. It is the engine that takes raw data and turns it into a model that can make predictions.

Here is the connection:

```
What you just did (DSPy):     Optimized PROMPTS (text) to improve a metric
What traditional ML does:     Optimizes WEIGHTS (numbers) to reduce a loss
```

The structure is identical — data goes in, predictions come out, a score tells you how well you did, and an optimizer updates the parameters to do better next time. The only difference is what the "parameters" are made of: text vs. numbers.

In the next section, we will break down the training loop step by step. By the end, you will see that everything you experienced with prompts and DSPy is a special case of this more general process.

## The Training Loop

The training loop is the heartbeat of machine learning. It is the process by which a model learns from data — adjusting its internal parameters (weights) step by step until it can make good predictions.

Every ML model you have ever heard of — image classifiers, speech recognizers, language models, recommendation systems — was trained using some version of this loop. Let's break it down.

### The Components

The training loop has seven key components. Each one plays a specific role:

**1. Inputs** — The raw data you feed into the model. Images, text, numbers, audio — whatever your task requires. For example, in our SQuAD demo, the inputs were context paragraphs and questions. In an image classifier, the inputs are pixel values.

**2. Labels** — The correct answers for each input. Also called "targets" or "ground truth." These are what you want the model to learn to predict. In SQuAD, the labels were the gold answers. In an image classifier, the label is the class name (e.g., "cat" or "dog").

**3. Weights** — The model's internal parameters — numbers that the model adjusts during training. A small model might have thousands of weights. GPT-4 has over a trillion. These weights determine how the model transforms inputs into predictions. At the start of training, weights are usually random — the model knows nothing.

**4. Predictions** — The model's output for a given input, computed using the current weights. At the start of training, predictions are essentially random guesses. As training progresses, they get closer to the correct labels.

**5. Loss Function** — A mathematical function that measures how far the model's predictions are from the correct labels. The loss is a single number: high means the model is doing badly, low means it is doing well. Common loss functions include Mean Squared Error (for regression) and Cross-Entropy (for classification). The loss is what the optimizer tries to minimize.

**6. Gradients** — The gradients tell us *how to change each weight to reduce the loss*. Technically, the gradient of the loss with respect to each weight is computed using backpropagation — a clever algorithm that works backwards through the model. You don't need to understand the math right now. Just know that gradients are the "direction of improvement" for each weight.

**7. Optimizer** — The algorithm that actually updates the weights using the gradients. The simplest optimizer is Stochastic Gradient Descent (SGD): `new_weight = old_weight - learning_rate * gradient`. More advanced optimizers like Adam adapt the learning rate for each weight. The optimizer is the engine that drives learning.

### The Loop

Here is how these components fit together in one training step:

```
                    ┌─────────────────────────────────────────────────┐
                    │              THE TRAINING LOOP                  │
                    └─────────────────────────────────────────────────┘

                              ┌──────────┐
                    ┌────────>│  Weights  │─────────┐
                    │         └──────────┘          │
                    │                               v
              ┌───────────┐                  ┌──────────────┐
              │ Optimizer │                  │  Predictions  │
              └───────────┘                  │  = Model(     │
                    ^                        │    Inputs,    │
                    │                        │    Weights)   │
              ┌───────────┐                  └──────────────┘
              │ Gradients │                         │
              │ (backprop)│                         v
              └───────────┘                  ┌──────────────┐
                    ^                        │     Loss      │
                    │                        │  = how wrong  │
                    └────────────────────────│  predictions  │
                                             │  vs. Labels   │
                                             └──────────────┘

     Inputs ──> Model(Weights) ──> Predictions ──> Loss ──> Gradients ──> Optimizer ──> Updated Weights
       ^                                                                                      │
       └──────────────────────────── repeat with next batch ──────────────────────────────────┘
```

One pass through this loop is called a **training step** (or iteration). The model sees a batch of data, makes predictions, computes the loss, calculates gradients, and updates its weights. Then it does it again with the next batch. And again. And again. Thousands or millions of times.

Each time around the loop, the weights get a little better. The loss goes down. The predictions get closer to the labels. That is learning.

### Batch, Epoch, and Iteration

Three terms you will hear constantly when people talk about training:

**Batch** — A small subset of the training data used in one training step. Instead of computing the loss on the entire dataset (which would be slow and memory-intensive), we compute it on a batch — typically 16, 32, 64, or 128 examples at a time. The gradients computed on a batch are an approximation of the true gradients, but they are good enough — and much faster to compute.

**Iteration** (or **step**) — One pass through the training loop using one batch. Feed a batch in, get predictions, compute loss, compute gradients, update weights. That is one iteration.

**Epoch** — One complete pass through the entire training dataset. If you have 10,000 training examples and your batch size is 100, then one epoch = 100 iterations. Training typically runs for multiple epochs — the model sees the same data multiple times, getting a little better each pass.

Here is a concrete example:

```
Training data:  10,000 examples
Batch size:     100
Iterations per epoch:  10,000 / 100 = 100
Number of epochs:      5
Total iterations:      100 x 5 = 500
```

So the model goes through the training loop 500 times total, seeing the full dataset 5 times. Each time through, the weights are updated and the model (hopefully) gets better.

```
┌─────────────────────────── Epoch 1 ───────────────────────────┐
│  Batch 1 → step    Batch 2 → step    ...    Batch 100 → step │
└───────────────────────────────────────────────────────────────┘
┌─────────────────────────── Epoch 2 ───────────────────────────┐
│  Batch 1 → step    Batch 2 → step    ...    Batch 100 → step │
└───────────────────────────────────────────────────────────────┘
                            ...                                  
┌─────────────────────────── Epoch 5 ───────────────────────────┐
│  Batch 1 → step    Batch 2 → step    ...    Batch 100 → step │
└───────────────────────────────────────────────────────────────┘
```

### Connecting Back to What You Did

Now you can see the full picture. When you were editing the signature docstring and re-running the evaluation, you were doing a human version of the training loop:

| **Training Loop** | **Your Manual Editing** | **BootstrapFewShot** |
|---|---|---|
| Inputs | SQuAD context + questions | SQuAD context + questions |
| Labels | Gold answers | Gold answers |
| Weights | Neural network numbers | Prompt text (instructions + demos) |
| Predictions | Model output | Model output |
| Loss / Metric | Answer hit rate | Answer hit rate |
| Gradients | Your intuition ("this prompt is too vague") | Optimizer's search strategy |
| Optimizer | You (edit prompt, re-run) | BootstrapFewShot |

The training loop is the universal pattern. Whether you are training a billion-parameter language model, fine-tuning an image classifier, or optimizing a prompt with DSPy — the structure is the same. Data goes in, predictions come out, a score tells you how well you did, and something updates the parameters to do better.

In the next sections, we will see this loop in action with traditional neural networks — training real weights (numbers) on real data, watching the loss go down and the predictions improve step by step.

## Demo: Neural Network Regression — Why Depth Matters

Time to see the training loop in action with real numbers.

We will generate **non-linear synthetic data** (a noisy sine wave) and try to fit it with two models:

1. **A single-layer linear model** — this can only learn a straight line. It will fail visibly.
2. **A multi-layer neural network** (2 hidden layers + ReLU activations) — this can learn curves. It will succeed.

Both models are built as PyTorch Lightning `LightningModule`s, so you can see exactly how the training loop components (model, `training_step`, `configure_optimizers`) map to the theory we just covered.

We will plot the model's predictions at multiple points during training so you can **watch the curve evolve** as the weights are updated.

In [ ]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset

# --- Generate non-linear synthetic data: noisy sine wave ---
torch.manual_seed(42)
x_data = torch.linspace(-3, 3, 200).unsqueeze(1)
y_data = torch.sin(x_data) + torch.randn_like(x_data) * 0.2

# Plot the raw data
plt.figure(figsize=(8, 4))
plt.scatter(x_data.numpy(), y_data.numpy(), s=10, alpha=0.6, label="Data (noisy sine)")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic Data: y = sin(x) + noise")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Data shape: x={list(x_data.shape)}, y={list(y_data.shape)}")
print("A straight line clearly cannot fit this curve.")

### Phase 1: Single-Layer Linear Model (Straight Line)

Our first model is the simplest possible neural network: a single linear layer with one input and one output. This is equivalent to fitting `y = wx + b` — a straight line.

We define it as a `LightningModule` so you can see the three key methods:
- `__init__` — define the model architecture (one linear layer)
- `training_step` — compute predictions and loss for one batch
- `configure_optimizers` — choose the optimizer (SGD)

In [ ]:
class LinearModel(pl.LightningModule):
    """Single-layer linear model: y = wx + b (a straight line)."""

    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(1, 1)
        self.snapshots = []  # store (epoch, loss, predictions) for plotting

    def forward(self, x):
        return self.linear(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.nn.functional.mse_loss(y_hat, y)
        return loss

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=0.01)


# Callback to capture snapshots during training
class SnapshotCallback(pl.Callback):
    """Capture model predictions at specific epochs for visualization."""

    def __init__(self, x_plot, snapshot_epochs):
        self.x_plot = x_plot
        self.snapshot_epochs = set(snapshot_epochs)

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch
        if epoch in self.snapshot_epochs:
            pl_module.eval()
            with torch.no_grad():
                preds = pl_module(self.x_plot).squeeze().cpu().numpy()
            # Get the current loss from the trainer's logged metrics
            loss = trainer.callback_metrics.get("train_loss", None)
            loss_val = loss.item() if loss is not None else None
            pl_module.snapshots.append((epoch, loss_val, preds))
            pl_module.train()


# We also need to log train_loss so the callback can read it
class LinearModelWithLog(LinearModel):
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.nn.functional.mse_loss(y_hat, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss


# Prepare data loader
dataset = TensorDataset(x_data, y_data)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Snapshot epochs for the linear model
linear_snapshot_epochs = [0, 9, 24, 49]

# Train the linear model
linear_model = LinearModelWithLog()
snapshot_cb = SnapshotCallback(x_data, linear_snapshot_epochs)

trainer = pl.Trainer(
    max_epochs=50,
    enable_progress_bar=True,
    enable_model_summary=False,
    enable_checkpointing=False,
    logger=False,
    callbacks=[snapshot_cb],
    accelerator="cpu",
)
trainer.fit(linear_model, loader)
print("Linear model training complete!")

In [ ]:
# Plot the linear model's predictions at each snapshot
x_np = x_data.numpy().squeeze()
y_np = y_data.numpy().squeeze()

fig, axes = plt.subplots(1, len(linear_model.snapshots), figsize=(4 * len(linear_model.snapshots), 4), sharey=True)
if len(linear_model.snapshots) == 1:
    axes = [axes]

for ax, (epoch, loss, preds) in zip(axes, linear_model.snapshots):
    ax.scatter(x_np, y_np, s=8, alpha=0.4, color="steelblue", label="Data")
    ax.plot(x_np, preds, color="red", linewidth=2, label="Linear model")
    loss_str = f"{loss:.4f}" if loss is not None else "N/A"
    ax.set_title(f"Epoch {epoch + 1} — Loss: {loss_str}")
    ax.set_xlabel("x")
    if ax == axes[0]:
        ax.set_ylabel("y")
    ax.legend(fontsize=8)

fig.suptitle("Linear Model: A Straight Line Cannot Fit a Curve", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("No matter how many epochs we train, the linear model can only produce a straight line.")
print("It minimizes the loss as best it can, but the fit is fundamentally limited.")
print("We need more layers — and non-linear activations — to fit curves.")

### Phase 2: Multi-Layer Neural Network (Curve Fitting)

Now we add **depth**. Our second model has:
- Input → 32 hidden units → ReLU → 32 hidden units → ReLU → 1 output

The ReLU activation (`max(0, x)`) is the key ingredient — it introduces **non-linearity**, allowing the network to learn curves instead of just straight lines.

We will train for more epochs and capture snapshots at multiple points so you can watch the curve progressively fit the data.

In [ ]:
class NonLinearModel(pl.LightningModule):
    """Multi-layer neural network: 2 hidden layers with ReLU."""

    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(1, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
        )
        self.snapshots = []

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.nn.functional.mse_loss(y_hat, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.01)


# Snapshot at multiple points to show progressive fitting
nn_snapshot_epochs = [0, 4, 14, 49, 99, 199]

nn_model = NonLinearModel()
nn_snapshot_cb = SnapshotCallback(x_data, nn_snapshot_epochs)

trainer_nn = pl.Trainer(
    max_epochs=200,
    enable_progress_bar=True,
    enable_model_summary=False,
    enable_checkpointing=False,
    logger=False,
    callbacks=[nn_snapshot_cb],
    accelerator="cpu",
)
trainer_nn.fit(nn_model, loader)
print("Non-linear model training complete!")

In [ ]:
# Plot the non-linear model's predictions at each snapshot
n_snaps = len(nn_model.snapshots)
cols = min(n_snaps, 3)
rows = (n_snaps + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), sharey=True)
axes_flat = [axes] if n_snaps == 1 else axes.flatten()

for idx, (epoch, loss, preds) in enumerate(nn_model.snapshots):
    ax = axes_flat[idx]
    ax.scatter(x_np, y_np, s=8, alpha=0.4, color="steelblue", label="Data")
    ax.plot(x_np, preds, color="darkorange", linewidth=2, label="NN prediction")
    loss_str = f"{loss:.4f}" if loss is not None else "N/A"
    ax.set_title(f"Epoch {epoch + 1} — Loss: {loss_str}")
    ax.set_xlabel("x")
    if idx % cols == 0:
        ax.set_ylabel("y")
    ax.legend(fontsize=8)

# Hide unused subplots
for idx in range(n_snaps, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Neural Network: Watch the Curve Fit the Data Over Training", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("With 2 hidden layers and ReLU activations, the network learns to fit the sine curve.")
print("Watch how the prediction evolves: random at first, then progressively matching the data.")
print("The loss decreases at each step — that is the training loop doing its job.")

### Key Takeaway: Why Depth Matters

The linear model (1 layer, no activation) can only learn straight lines — no matter how long you train it. The multi-layer network (2 hidden layers + ReLU) can learn curves, because:

1. **More layers** = more capacity to represent complex functions
2. **Non-linear activations (ReLU)** = the ability to bend and curve, not just draw straight lines

This is the core insight behind **deep learning**: stacking layers with non-linear activations lets neural networks approximate virtually any function. The training loop finds the right weights to make it happen.

```
Linear model:      Input ──> [Linear] ──> Output          (can only learn: y = wx + b)
Neural network:    Input ──> [Linear → ReLU → Linear → ReLU → Linear] ──> Output
                                                           (can learn: any smooth curve)
```

## Demo: FoodyDudy Image Classification

Now let's see the training loop applied to a real-world image classification task.

We will use the [FoodyDudy](https://github.com/GemmyTheGeek/FoodyDudy) dataset — **48 classes of Thai food**, with ~240 training images per class (11,520 total). This dataset was created by [@GemmyTheGeek](https://github.com/GemmyTheGeek), an AI Builders alumnus (Class of 2021).

Our approach:

1. Clone the dataset
2. Create a mini validation set (5 images per class = 240 images) for fast evaluation
3. Rename folders from numeric IDs to English food names
4. Use **fastai** with a pretrained **ResNet34** to fine-tune on this data
5. Print accuracy on the validation set

This demo shows the same training loop concepts in a high-level framework:
- **Data loading** = `DataBlock` + `dataloaders`
- **Pretrained model** = initial weights (ResNet34 trained on ImageNet)
- **Fine-tuning** = the training loop updating those weights for our task
- **Accuracy** = the metric we track

### Step 1: Clone the Dataset and Create Mini Validation Set

We clone the FoodyDudy repository, then create a `valid_mini` folder with 5 images per class for fast validation.

In [ ]:
import os
import glob
import shutil
import subprocess

# Clone the FoodyDudy dataset
if not os.path.exists("FoodyDudy"):
    print("Cloning FoodyDudy dataset (this may take a few minutes)...")
    subprocess.run(["git", "clone", "https://github.com/GemmyTheGeek/FoodyDudy.git"], check=True)
    print("Done!")
else:
    print("FoodyDudy already exists, skipping clone.")

# Create mini validation set: 5 images per class
valid_mini_dir = "FoodyDudy/images/valid_mini"
if os.path.exists(valid_mini_dir):
    shutil.rmtree(valid_mini_dir)
os.makedirs(valid_mini_dir, exist_ok=True)

valid_folders = sorted(glob.glob("FoodyDudy/images/valid/*"))
valid_fnames = []
for folder in valid_folders:
    valid_fnames += sorted(glob.glob(f"{folder}/*"))[:5]

for i in range(48):
    os.makedirs(f"{valid_mini_dir}/{str(i).zfill(2)}", exist_ok=True)

for fname in valid_fnames:
    class_id = fname.split("/")[-2]
    basename = fname.split("/")[-1]
    shutil.copyfile(fname, f"{valid_mini_dir}/{class_id}/{basename}")

print(f"Mini validation set created: {len(valid_fnames)} images in {valid_mini_dir}")

### Step 2: Rename Folders to English Food Names

The original folders are named `00`, `01`, ..., `47`. We rename them to human-readable English names so the model's predictions are easier to interpret.

In [ ]:
# English names for the 48 Thai food classes
food_d = [
    'green_curry', 'tepo_curry', 'liang_curry', 'taohoo_moosup', 'mara_yadsai',
    'masaman', 'orange_curry', 'cashew_chicken', 'omelette', 'sunny_side_up',
    'palo_egg', 'sil_egg', 'nun_banana', 'kua_gai', 'cabbage_fish_sauce',
    'river_prawn', 'shrimp_ob_woonsen', 'kanom_krok', 'mango_sticky_rice', 'kao_kamoo',
    'kao_klook_kapi', 'kaosoi', 'kao_pad', 'kao_pad_shrimp', 'chicken_rice',
    'kao_mok_gai', 'tom_ka_gai', 'tom_yum_kung', 'tod_mun', 'poh_pia',
    'pak_boong_fai_daeng', 'padthai', 'pad_krapao', 'pad_si_ew', 'pad_fakthong',
    'eggplant_stirfry', 'pad_hoi_lai', 'foithong', 'panaeng', 'yum_tua_ploo',
    'yum_woonsen', 'larb_moo', 'pumpkin_custard', 'sakoo_sai_moo', 'somtam',
    'moopoing', 'satay', 'hor_mok'
]

# Rename train and valid_mini folders
for split in ['train', 'valid_mini']:
    for i in range(48):
        old_name = f"FoodyDudy/images/{split}/{str(i).zfill(2)}"
        new_name = f"FoodyDudy/images/{split}/{str(i).zfill(2)}_{food_d[i]}"
        if os.path.exists(old_name):
            os.rename(old_name, new_name)

# Show a few renamed folders
train_folders = sorted(os.listdir("FoodyDudy/images/train"))[:5]
print("Sample train folders (first 5):")
for f in train_folders:
    print(f"  {f}")
print(f"\nTotal classes: {len(os.listdir('FoodyDudy/images/train'))}")

### Step 3: Build the DataBlock and Fine-Tune with fastai

We use fastai's `DataBlock` API to set up data loading, then create a `vision_learner` with a pretrained **ResNet34** backbone. Fine-tuning means we start with weights that already know how to recognize visual features (edges, textures, shapes) from ImageNet, and we adapt them to recognize Thai food.

This is transfer learning — one of the most powerful ideas in deep learning.

In [ ]:
from fastai.vision.all import *

# Define the DataBlock
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=GrandparentSplitter(valid_name='valid_mini'),
    get_y=parent_label,
    batch_tfms=aug_transforms(size=224)
)

dls = dblock.dataloaders('FoodyDudy/images/', bs=64)

print(f"Training batches: {len(dls.train)}")
print(f"Validation batches: {len(dls.valid)}")
print(f"Number of classes: {dls.c}")
print(f"Class names (first 10): {dls.vocab[:10]}")

In [ ]:
# Create a vision learner with pretrained ResNet34
learn = vision_learner(dls, resnet34, metrics=accuracy)

# Fine-tune for 2 epochs (short for demo purposes)
print("Fine-tuning ResNet34 on FoodyDudy (2 epochs)...")
print("(This trains the head first, then unfreezes and trains the full model)\n")
learn.fine_tune(2)

print("\nTraining complete!")

### Step 4: Check Accuracy and Show Sample Predictions

Let's see how well the model does on our mini validation set, and look at a few predictions.

In [ ]:
# Print validation accuracy
val_results = learn.validate()
val_loss, val_accuracy = val_results[0], val_results[1]
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.1f}%)")

# Show some predictions
print("\nSample predictions on validation set:")
learn.show_results(max_n=9, figsize=(10, 10))

### Connecting Back to the Training Loop

Even though fastai hides most of the details behind a clean API, the same training loop is running under the hood:

| **Training Loop Concept** | **What fastai does** |
|---|---|
| **Data loading** (Inputs + Labels) | `DataBlock` + `dataloaders` — loads images in batches of 64, with augmentation |
| **Pretrained model** (Initial Weights) | `resnet34` — starts with weights pre-trained on ImageNet (1.2M images, 1000 classes) |
| **Fine-tuning** (The Training Loop) | `fine_tune(2)` — first trains only the new head, then unfreezes all layers and trains end-to-end |
| **Accuracy** (Metric) | `metrics=accuracy` — tracks what fraction of predictions match the true labels |

The key insight: we did not train from scratch. We started with a model that already understands visual features (edges, textures, shapes) and adapted it to our specific task. This is **transfer learning** — and it is why you can get good results on 48 food classes with just 2 epochs of training.

```
ImageNet weights (general vision) ──> Fine-tune on FoodyDudy ──> Thai food classifier
         (pretrained)                    (training loop)              (adapted)
```

## Evaluation: Metrics, Splits, and Beyond

We have now trained three different systems: an LLM System (manual + DSPy), a neural network regression model, and an image classifier. In every case, we needed a way to answer the question: **how good is this model?**

That is what evaluation is about. Let's formalize the key concepts.

### Loss vs. Metric

These two terms sound similar but serve different purposes:

**Loss** is the function the training loop optimizes. It must be differentiable (so we can compute gradients) and it measures how far the model's predictions are from the correct labels. The optimizer's job is to make the loss go down. Examples: Mean Squared Error (MSE), Cross-Entropy.

**Metric** is the number *you* care about — the human-readable measure of how well the model performs on the task. It does not need to be differentiable. It just needs to be meaningful. Examples: accuracy, F1 score, answer hit rate.

Sometimes the loss and the metric are the same thing (e.g., MSE for regression). Often they are not. In our FoodyDudy demo, the loss was cross-entropy (used for gradient updates) but the metric was accuracy (what we reported). In the SQuAD demo, DSPy optimized answer hit rate directly — but there was no gradient-based loss at all, because prompt optimization does not use gradients.

| | **Loss** | **Metric** |
|---|---|---|
| **Purpose** | Guide the optimizer during training | Report performance to humans |
| **Must be differentiable?** | Yes (for gradient-based training) | No |
| **Who uses it?** | The optimizer | You (the developer / researcher) |
| **Examples** | MSE, Cross-Entropy, Binary Cross-Entropy | Accuracy, Precision, Recall, F1, MAE |

### Common Metrics

Different tasks call for different metrics. Here are the most common ones:

**Classification metrics** (predicting a category):

- **Accuracy** — fraction of predictions that are correct. Simple and intuitive, but misleading when classes are imbalanced (e.g., 99% of emails are not spam → a model that always says "not spam" gets 99% accuracy).
- **Precision** — of all the items the model predicted as positive, how many actually were? High precision = few false alarms.
- **Recall** — of all the items that actually were positive, how many did the model catch? High recall = few missed cases.
- **F1 Score** — the harmonic mean of precision and recall. Balances both concerns. Useful when you care about both false positives and false negatives.

**Regression metrics** (predicting a number):

- **MSE (Mean Squared Error)** — average of squared differences between predictions and true values. Penalizes large errors heavily.
- **MAE (Mean Absolute Error)** — average of absolute differences. More robust to outliers than MSE.

Let's see these in action with tiny toy examples.

#### Toy Example: Classification Metrics with a Confusion Matrix

Imagine a spam classifier evaluated on 10 emails. We build a confusion matrix and compute accuracy, precision, recall, and F1 by hand.

In [ ]:
import numpy as np

# --- 10 emails: ground truth and model predictions ---
# 1 = spam, 0 = not spam
y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])  # 4 spam, 6 not-spam
y_pred = np.array([1, 1, 0, 0, 0, 0, 0, 0, 1, 0])  # model's guesses

# --- Confusion matrix (by hand) ---
TP = int(((y_true == 1) & (y_pred == 1)).sum())  # true positives
FP = int(((y_true == 0) & (y_pred == 1)).sum())  # false positives
FN = int(((y_true == 1) & (y_pred == 0)).sum())  # false negatives
TN = int(((y_true == 0) & (y_pred == 0)).sum())  # true negatives

print("Confusion Matrix (10 emails):")
print(f"                 Predicted Spam   Predicted Not-Spam")
print(f"  Actual Spam         TP={TP}              FN={FN}")
print(f"  Actual Not-Spam     FP={FP}              TN={TN}")
print()

# --- Metrics ---
accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy  = (TP+TN) / total       = ({TP}+{TN}) / 10 = {accuracy:.2f}")
print(f"Precision = TP / (TP+FP)           = {TP} / ({TP}+{FP}) = {precision:.2f}")
print(f"Recall    = TP / (TP+FN)           = {TP} / ({TP}+{FN}) = {recall:.2f}")
print(f"F1        = 2*P*R / (P+R)          = {f1:.2f}")
print()
print("Notice: accuracy is 70% (looks okay), but recall is only 50% — ")
print("the model missed half the spam. F1 balances both concerns.")

#### Toy Example: MSE vs MAE — What Happens with an Outlier?

MSE squares the errors, so one large mistake dominates the score. MAE just takes the absolute value, making it more robust. Let's see the difference.

In [ ]:
import numpy as np

# --- Case 1: No outlier (predictions are close to truth) ---
y_true_1 = np.array([3.0, 5.0, 2.5, 7.0, 4.0])
y_pred_1 = np.array([2.8, 5.2, 2.3, 7.1, 4.2])

errors_1 = y_pred_1 - y_true_1
mse_1 = (errors_1 ** 2).mean()
mae_1 = np.abs(errors_1).mean()

print("=== Case 1: No outlier ===")
print(f"  True:       {y_true_1}")
print(f"  Predicted:  {y_pred_1}")
print(f"  Errors:     {errors_1}")
print(f"  MSE = {mse_1:.4f}    MAE = {mae_1:.4f}")
print()

# --- Case 2: One outlier (one prediction is way off) ---
y_true_2 = np.array([3.0, 5.0, 2.5, 7.0, 4.0])
y_pred_2 = np.array([2.8, 5.2, 2.3, 7.1, 14.0])  # last prediction is 10 off!

errors_2 = y_pred_2 - y_true_2
mse_2 = (errors_2 ** 2).mean()
mae_2 = np.abs(errors_2).mean()

print("=== Case 2: One outlier (last prediction is 10 off) ===")
print(f"  True:       {y_true_2}")
print(f"  Predicted:  {y_pred_2}")
print(f"  Errors:     {errors_2}")
print(f"  MSE = {mse_2:.4f}    MAE = {mae_2:.4f}")
print()

print(f"MSE jumped from {mse_1:.4f} to {mse_2:.4f} ({mse_2/mse_1:.0f}x increase!)")
print(f"MAE jumped from {mae_1:.4f} to {mae_2:.4f} ({mae_2/mae_1:.1f}x increase)")
print()
print("MSE squares the error, so the outlier (10^2 = 100) dominates the score.")
print("MAE just takes |10| = 10, so the impact is proportional, not explosive.")
print("Choose MSE when large errors are especially bad. Choose MAE when you want robustness.")

### Train, Validation, and Test Splits

When you train a model, you need to know: does it actually work on *new* data it has never seen? Or did it just memorize the training examples?

To answer this, we split our data into three parts:

- **Training set** — the data the model learns from. Weights are updated based on this data.
- **Validation set** — data the model never trains on, used to tune hyperparameters (learning rate, number of layers, prompt wording) and monitor for overfitting during development.
- **Test set** — data held out until the very end. You evaluate on it *once* to get a final, unbiased estimate of performance. If you peek at the test set during development, it is no longer unbiased.

```
┌──────────────────────────────────────────────────────────────────┐
│                        ALL YOUR DATA                            │
├────────────────────────────────┬──────────────┬─────────────────┤
│         Training Set           │  Validation   │    Test Set     │
│          (60-80%)              │   (10-20%)    │    (10-20%)     │
│                                │              │                 │
│  Model learns from this data.  │  Used during  │  Held out until │
│  Weights are updated here.     │  development  │  the very end.  │
│                                │  to tune      │  Evaluate once  │
│                                │  choices and  │  for final      │
│                                │  catch        │  unbiased       │
│                                │  overfitting. │  score.         │
├────────────────────────────────┴──────────────┴─────────────────┤
│  RULE: No data point may appear in more than one split.         │
│  If training data leaks into validation or test, your scores    │
│  will be artificially high and your model will fail in the      │
│  real world. This is called DATA LEAKAGE.                       │
└──────────────────────────────────────────────────────────────────┘
```

**Why no overlap?** If the model has already seen a data point during training, it may have memorized the answer. Evaluating on that same point tells you nothing about how the model handles *new* inputs. The whole point of evaluation is to estimate real-world performance — and in the real world, the model will see data it has never encountered before.

### But What About SQuAD?

The metrics above work well when the expected output is clean — a class label, a number, a yes/no. But remember the SQuAD demo? The answers were **free-form text**.

We ran into this problem directly:

- **Exact string match** was too strict. The gold answer might be `"the United States"` but the model says `"Yes, they are both from the United States"` — that is clearly correct, but exact match gives it a score of zero.
- So we used **answer hit rate** — check if the gold answer appears anywhere in the model's response. This is more forgiving and works well for short factual answers.

But even answer hit has limits. What if the model says `"America"` instead of `"the United States"`? Semantically correct, but our simple substring check misses it. What about tasks where the answer is a full paragraph — a summary, an explanation, a creative response? Substring matching becomes meaningless.

This is the fundamental challenge of **evaluating free-text outputs**: there is no single number that perfectly captures "how good is this text?"

```
Gold answer:    "the United States"
Prediction 1:   "They are both from the United States"  → Hit: YES  ✓
Prediction 2:   "America"                               → Hit: NO   ✗ wrong score!
Prediction 3:   "the United States"                     → Hit: YES  ✓ perfect
```

### The Frontier: Evaluating LLM Systems

As LLM systems produce increasingly complex free-text outputs — summaries, code, explanations, conversations — the evaluation problem gets harder. The field is actively developing new approaches:

- **LLM-as-Judge** — use a (usually stronger) LLM to evaluate the output of another LLM. You give the judge the question, the gold answer, and the model's response, and ask it to score the response. Surprisingly effective, but introduces its own biases.
- **Human Evaluation** — have humans rate the outputs. The gold standard for quality, but expensive, slow, and hard to scale.
- **Rubric-based Scoring** — define a detailed rubric (e.g., "Does the answer contain the key entity? Is it factually correct? Is it concise?") and score each criterion separately, either by humans or by an LLM judge.

We will not implement any of these here — that is a full lesson on its own. But it is important to know they exist, because if you build an LLM system for anything beyond simple factoid QA, you will need them.

We will cover evaluation in depth — including hands-on implementation of these modern approaches — in the dedicated metrics lesson:

> **Next up:** [`notebooks/03a_metrics_and_baselines.ipynb`](03a_metrics_and_baselines.ipynb) — Metrics, Baselines, and Modern Evaluation

## Vocabulary Review

Here are the key terms from this notebook. Make sure you can explain each one in your own words.

### Traditional ML Terms

| Term | Definition |
|---|---|
| **Inputs** | The raw data fed into a model (images, text, numbers, etc.) |
| **Labels** | The correct answers for each input, also called targets or ground truth |
| **Weights** | The model's internal learnable parameters — numbers adjusted during training |
| **Loss** | A function measuring how far predictions are from labels; the optimizer minimizes this |
| **Gradients** | The direction and magnitude of change needed for each weight to reduce the loss |
| **Optimizer** | The algorithm that updates weights using gradients (e.g., SGD, Adam) |
| **Epoch** | One complete pass through the entire training dataset |
| **Batch** | A small subset of training data used in one training step |
| **Metric** | A human-readable measure of model performance (e.g., accuracy, F1, MAE) |
| **Training Set** | Data the model learns from — weights are updated based on this data |
| **Validation Set** | Held-out data used during development to tune hyperparameters and detect overfitting |
| **Test Set** | Data reserved for final evaluation — used once to get an unbiased performance estimate |

### 2026 Additions

| Term | Definition |
|---|---|
| **LLM (Large Language Model)** | A large-scale neural network trained on text to predict the next token (e.g., GPT, Qwen, Llama) |
| **LLM System** | An AI system built on a pre-trained LLM, using prompts to perform tasks — closer to software engineering than traditional ML |
| **Prompt Optimization** | Automatically searching for better prompts (instructions, few-shot examples) to improve LLM performance on a task — the LLM-era equivalent of training weights |
| **LLM-as-Judge** | Using an LLM to evaluate the output of another LLM, as an alternative to traditional metrics for free-text evaluation |

## Reflection Questions

Take a few minutes to think about these questions. There are no single right answers — the goal is to connect what you learned to your own thinking.

**1. Three paradigms in the wild.** Think of three AI-powered products or features you use in daily life. For each one, which paradigm do you think it uses — rule-based, traditional ML, or LLM system? What clues help you decide? (Hint: consider whether the system seems to follow fixed rules, whether it was likely trained on labeled data, or whether it feels like it is "understanding" free-form input.)

**2. When to choose ML vs. LLM.** Imagine you are building a system to detect fraudulent credit card transactions in real time. Would you use a traditional ML model or an LLM system? What about a system that summarizes customer support tickets? For each case, explain your reasoning — consider factors like latency, data availability, output format, and cost.

**3. Evaluating an LLM system.** Suppose you built an LLM system that generates study notes from lecture transcripts. How would you evaluate whether the notes are good? Exact match clearly will not work. Answer hit rate probably will not either. What evaluation strategy would you design? Would you use LLM-as-judge, human evaluation, rubrics, or some combination?

**4. Manual vs. automatic optimization.** You tried writing prompts by hand, then watched DSPy optimize prompts automatically. Compare the two experiences. What strategies did you try manually? What did DSPy's optimizer find that you did not? If DSPy beat your score, why do you think the automated search found something better? If you beat DSPy, what human intuition did you bring that the optimizer lacked?